In [ ]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
%%writefile filter.sh

# Calculate variant and sample missingness
plink2 \
    --bfile "${GENO}/EUR_ab0.35_dp20_chr${CHROM}" \
    --missing \
    --memory 3100 \
    --out "${OUTPUT}/EUR_ab0.35_dp20_call0.9_hwe1e15_chr${CHROM}"

# Apply variant-level quality control filters and export the filtered dataset
plink2 \
    --bfile "${GENO}/EUR_ab0.35_dp20_chr${CHROM}" \
    --hwe 1e-15 \
    --geno 0.1 \
    --mac 1 \
    --memory 3100 \
    --make-bed \
    --out "${OUTPUT}/EUR_ab0.35_dp20_call0.9_hwe1e15_chr${CHROM}"


In [ ]:
# Define path to logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Define path to output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink"
# Define mount path to genomic data 
MNT_PATH="/mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/chip/plink"
# Define virtual machine disk size 
DISK_SIZE = 4 
    
# Loop through chromosomes 
for CHROM in [2, 4, 15, 15]:    
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    # Define path to genomic data 
    BED=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr{CHROM}.bed'
    BIM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr{CHROM}.bim'
    FAM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr{CHROM}.fam'
    
    # Sumbission command for dsub 
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BED={BED}",
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "filter.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    
    print(dsub_command.stdout)
    print(dsub_command.stderr)